In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
from astropy import units as u
from astropy import constants as const
import matplotlib.pyplot as plt
from functions import *
import multiple_planets_gas_acc as code_gas


In [ ]:
r0 = np.arange(50, 200, 1)
time =0.1
params = code_gas.Params()
timescale_disc = t_s(r0,time, params)
fig,axs = plt.subplots(1,1)
axs.loglog(r0, timescale_disc)
axs.set_xlabel("r0 [au]")
axs.set_ylabel("t_s [Myr]")

In [ ]:
import time
import numpy as np
import scipy.integrate as si
if not hasattr(si, "cumtrapz"):
    si.cumtrapz = si.cumulative_trapezoid

import multiple_planets_gas_acc as code

params = code.Params(H_r_model="Lambrechts_mixed", star_mass=0.2*code.const.M_sun.to(code.u.M_earth).value, Z=0.01)
sim_params = code.SimulationParams(N_step=100, m0=np.array([1e-3]), a_p0=np.array([20.0]), t0=np.array([0.1]), t_fin=0.2)
peb_acc = code.PebbleAccretion(simplified_acc=True)
gas_acc = code.GasAccretion()

# warmup
code.M_dot_star(0.1, params)

t0 = time.perf_counter()
for _ in range(10000):
    code.M_dot_star(0.1, params)
t1 = time.perf_counter()

print("M_dot_star avg:", (t1 - t0) / 10000)

t0 = time.perf_counter()
code.simulate_euler(True, True, peb_acc, gas_acc, params, sim_params, output_folder="sims/gas_acc")
t1 = time.perf_counter()

print("simulate_euler total:", t1 - t0)

In [ ]:
import statistics as stats
import tempfile
from pathlib import Path

import scipy.integrate as si
if not hasattr(si, "cumtrapz"):
    si.cumtrapz = si.cumulative_trapezoid

baseline_mdot = 1.6081391599982453e-05
baseline_sim = 0.028741500000251108

bench_dir = Path(tempfile.mkdtemp(prefix="gas_acc_bench_"))


def bench_mdot(n=10000, reps=10):
    values = []
    code.M_dot_star(0.1, params)
    for _ in range(reps):
        t0 = time.perf_counter()
        for _ in range(n):
            code.M_dot_star(0.1, params)
        t1 = time.perf_counter()
        values.append((t1 - t0) / n)
    return values


def bench_sim(reps=5):
    values = []
    for _ in range(reps):
        t0 = time.perf_counter()
        code.simulate_euler(True, True, peb_acc, gas_acc, params, sim_params, output_folder=str(bench_dir))
        t1 = time.perf_counter()
        values.append(t1 - t0)
    return values

mdot_times = bench_mdot()
sim_times = bench_sim()

mdot_median = stats.median(mdot_times)
sim_median = stats.median(sim_times)

print(f"M_dot_star median: {mdot_median:.6e} s")
print(f"M_dot_star change vs baseline: {(baseline_mdot - mdot_median) / baseline_mdot * 100:.2f}%")
print(f"simulate_euler median: {sim_median:.6e} s")
print(f"simulate_euler change vs baseline: {(baseline_sim - sim_median) / baseline_sim * 100:.2f}%")


In [ ]:
import statistics as stats

import scipy.integrate as si
if not hasattr(si, "cumtrapz"):
    si.cumtrapz = si.cumulative_trapezoid

# Next hotspot from the line profiler: st_frag_drift -> st_drift -> mfp/rho_0/
# Compare against a saved baseline once you have it.

position = 20.0
mdot_star = code.M_dot_star(0.1, params)
H_r = code.H_R(position, mdot_star, params)
sigma_gas = code.sigma_gas_steady_state(position, H_r, mdot_star, params)


def bench_st_frag_drift(n=2000, reps=10):
    values = []
    code.st_frag_drift(position, mdot_star, H_r, sigma_gas, params)
    for _ in range(reps):
        t0 = time.perf_counter()
        for _ in range(n):
            code.st_frag_drift(position, mdot_star, H_r, sigma_gas, params)
        t1 = time.perf_counter()
        values.append((t1 - t0) / n)
    return values


st_times = bench_st_frag_drift()
print("st_frag_drift median:", stats.median(st_times))
print("st_frag_drift mean:", stats.mean(st_times))
print("st_frag_drift stdev:", stats.pstdev(st_times))

In [ ]:
import statistics as stats

# Next layer down from st_frag_drift: isolate the helper calls it depends on.
position = 20.0
mdot_star = code.M_dot_star(0.1, params)
H_r = code.H_R(position, mdot_star, params)
sigma_gas = code.sigma_gas_steady_state(position, H_r, mdot_star, params)


def bench_func(func, args, n=20000, reps=8):
    values = []
    func(*args)
    for _ in range(reps):
        t0 = time.perf_counter()
        for _ in range(n):
            func(*args)
        t1 = time.perf_counter()
        values.append((t1 - t0) / n)
    return values


helper_benchmarks = {
    "omega_k": bench_func(code.omega_k, (position, params)),
    "v_k": bench_func(code.v_k, (position, params)),
    "rho_0": bench_func(code.rho_0, (position, H_r, sigma_gas)),
    "mfp": bench_func(code.mfp, (position, H_r, sigma_gas, params)),
    "st_drift_epstein": bench_func(code.st_drift_epstein, (position, H_r, mdot_star, sigma_gas, params)),
    "st_drift_stokes": bench_func(code.st_drift_stokes, (position, H_r, mdot_star, sigma_gas, params)),
}

for name, values in helper_benchmarks.items():
    print(f"{name} median: {stats.median(values):.6e} s")
    print(f"{name} mean: {stats.mean(values):.6e} s")
    print(f"{name} stdev: {stats.pstdev(values):.6e} s")
    print()


In [ ]:
import statistics as stats

# Benchmark the current omega_k against a float-only version with cached coefficients.
position = 20.0
omega_coeff = np.sqrt(code.G * params.star_mass)


def omega_k_raw(pos):
    return np.sqrt(code.G * params.star_mass / pos**3)


def omega_k_cached(pos):
    return omega_coeff / pos**1.5


def bench_func(func, n=50000, reps=10):
    values = []
    func(position)
    for _ in range(reps):
        t0 = time.perf_counter()
        for _ in range(n):
            func(position)
        t1 = time.perf_counter()
        values.append((t1 - t0) / n)
    return values


omega_current = bench_func(lambda pos: code.omega_k(pos, params))
omega_raw = bench_func(omega_k_raw)
omega_cached = bench_func(omega_k_cached)

print(f"omega_k current median: {stats.median(omega_current):.6e} s")
print(f"omega_k raw median: {stats.median(omega_raw):.6e} s")
print(f"omega_k cached median: {stats.median(omega_cached):.6e} s")
